# Groundwater transport (GWT) model

This notebook builds and runs a groundwater transport (GWT) simulation for the synthetic valley. It reuses the heads and flows from the groundwater flow (GWF) model that is built and run in `flopy-intro-gwt-A.ipynb`, so **run that notebook first**.

## Imports and helper functions

Import FloPy and the plotting libraries used throughout the notebook. The `time_slider_view` helper draws a single interactive figure that updates as you move a time slider; it renders each frame to a PNG so exactly one figure is shown in both VS Code and JupyterLab.

In [ ]:
%matplotlib inline
import datetime
import io
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Image, IntSlider
from IPython.display import display
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
# Drive a time-index slider that always shows exactly one figure, identically
# in VS Code and JupyterLab. Each frame is rendered to a PNG and pushed into a
# single ipywidgets.Image; assigning Image.value *replaces* the displayed frame
# in every frontend, so figures never stack. `make_fig(time_index)` must build
# and return a Matplotlib figure.
def time_slider_view(make_fig, ntimes, value=None, description="time index", dpi=200):
    image = Image(format="png")
    slider = IntSlider(
        min=0,
        max=ntimes - 1,
        step=1,
        value=ntimes - 1 if value is None else value,
        description=description,
        continuous_update=False,
    )

    def update(*_):
        fig = make_fig(slider.value)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi)
        plt.close(fig)  # keep the inline backend from also emitting the figure
        image.value = buf.getvalue()

    slider.observe(update, names="value")
    display(slider, image)
    update()

## Load the existing flow model

Load the flow model from the `models/` workspace where notebook A wrote and ran it. We read a few quantities from it (layer and cell counts, the number of stress periods, and the center of the lake) that the transport model and figures need.

In [ ]:
# the gwf model was built and run in flopy-intro-gwt-A.ipynb; load that
# already-run model from its models/ workspace

name = "sv"
new_ws = pl.Path("models/synthetic-valley-vg")
gwt_ws = new_ws.parent / "synthetic-valley-vg-gwt"

In [ ]:
sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=new_ws, write_headers=False
)

In [ ]:
gwf = sim.get_model(name)
nlay, ncpl = gwf.disv.nlay.array, gwf.disv.ncpl.array
shape3d = (nlay, ncpl)

nper = sim.tdis.nper.array
perlen = sim.tdis.perioddata.array["perlen"]

# x-coordinate of the center of the lake (LAK) package; the model cross
# sections (A-A') are taken along this north-south line so they cut through
# the middle of the lake instead of the center of the model domain
lak_cellids = gwf.lak.connectiondata.array["cellid"]
lak_icpl = np.array(sorted({c[-1] for c in lak_cellids}))
lake_center_x = float(gwf.modelgrid.xcellcenters[lak_icpl].mean())

## Check the flow model has been run

The transport model reads heads and cell-by-cell flows directly from the flow model, so that output must already exist. This check raises a clear error if the head or budget file is missing, empty, or contains no time steps, pointing you back to notebook A.

In [ ]:
# the transport model reads heads and flows from the gwf model, so the gwf
# output must already exist. fail early with a clear message if the head or
# budget (cell-by-cell) file is missing or empty.
def require_gwf_output(gwf, ws):
    checks = {
        "head": (gwf.oc.head_filerecord.array, gwf.output.head),
        "budget": (gwf.oc.budget_filerecord.array, gwf.output.budget),
    }
    for rtype, (filerecord, reader) in checks.items():
        if filerecord is None:
            raise FileNotFoundError(
                f"gwf OC has no {rtype} output configured; run "
                "flopy-intro-gwt-A.ipynb to run the gwf model first"
            )
        fpath = ws / filerecord[0][0]
        if not fpath.is_file() or fpath.stat().st_size == 0:
            raise FileNotFoundError(
                f"gwf {rtype} output '{fpath}' is missing or empty; run "
                "flopy-intro-gwt-A.ipynb to run the gwf model first"
            )
        if not reader().get_times():
            raise ValueError(
                f"gwf {rtype} output '{fpath}' contains no times; re-run "
                "the gwf model in flopy-intro-gwt-A.ipynb"
            )


require_gwf_output(gwf, new_ws)

## Build the transport model

Set up the GWT model on the same grid as the flow model. The flow-model interface (FMI) package feeds it the saved heads and flows, the source-sink mixing (SSM) package introduces solute from the lake, and the advection, dispersion, and storage packages control how the solute spreads.

### Create the simulation and solver

Create a new simulation for the transport model, copy the flow model's time discretization, and configure the iterative solver.

In [ ]:
gwt_name = f"{name}_gwt"

In [ ]:
sim_gwt = flopy.mf6.MFSimulation(
    sim_name=gwt_name,
    sim_ws=gwt_ws,
    # continue_=True,
)

perioddata = sim.tdis.perioddata.array
# perioddata["nstp"][1:] = 20
# perioddata["tsmult"][1:] = 1.0
tdis = flopy.mf6.ModflowTdis(
    sim_gwt,
    time_units="DAYS",
    nper=nper,
    perioddata=perioddata,
)
ims = flopy.mf6.ModflowIms(
    sim_gwt,
    filename=f"{gwt_name}.ims",
    print_option="all",
    complexity="simple",
    outer_maximum=500,
    inner_maximum=200,
    linear_acceleration="bicgstab",
)

### Add the transport model and packages

Add the GWT model and its packages: grid (DISV), initial concentration, advection, dispersion, mobile storage, the flow interface, the lake solute source, and output control.

In [ ]:
gwt = flopy.mf6.ModflowGwt(
    sim_gwt,
    modelname=gwt_name,
    # dependent_variable_scaling=True,
)
dis = flopy.mf6.ModflowGwtdisv(
    gwt,
    length_units="meters",
    nlay=nlay,
    ncpl=ncpl,
    cell2d=gwf.disv.cell2d.array,
    nvert=gwf.disv.nvert.array,
    vertices=gwf.disv.vertices.array,
    top=gwf.disv.top.array,
    botm=gwf.disv.botm.array,
    idomain=gwf.disv.idomain.array,
)
ic = flopy.mf6.ModflowGwtic(gwt, strt=0.0)
adv = flopy.mf6.ModflowGwtadv(
    gwt,
    # ats_percel=0.75,
    scheme="utvd",
    # scheme="upstream",
)
dsp = flopy.mf6.ModflowGwtdsp(gwt, diffc=0.0e-12, alh=75.0, ath1=7.5)
mst = flopy.mf6.ModflowGwtmst(
    gwt,
    porosity=gwf.sto.sy.array,
    # zero_order_decay=True,
    # decay=-1.0 / 365.25,
)
pd = [
    ("GWFHEAD", f"../synthetic-valley-vg/{name}.hds", None),
    ("GWFBUDGET", f"../synthetic-valley-vg/{name}.cbc", None),
]
fmi = flopy.mf6.ModflowGwtfmi(gwt, packagedata=pd)
sourcerecarray = [
    ("LAK-1", "AUX", "CONCENTRATION"),
]
ssm = flopy.mf6.ModflowGwtssm(gwt, sources=sourcerecarray)
oc = flopy.mf6.ModflowGwtoc(
    gwt,
    concentration_filerecord=f"{name}.ucn",
    saverecord={
        1: [
            ("CONCENTRATION", "LAST"),
        ]
    },
    printrecord=[("BUDGET", "ALL")],
)

# add mover package
# mvt = flopy.mf6.ModflowGwtmvt(gwt)

## Run the transport model

Write the transport input files and run the simulation.

In [ ]:
sim_gwt.write_simulation()

In [ ]:
sim_gwt.run_simulation()

## View the simulated concentrations

Plot the simulated solute concentrations. Use the time slider to step through the simulation.

### Inspect the concentration output

A quick look at the concentration output: the range of simulated values and the data at the first output time.

In [ ]:
times = gwt.output.concentration().get_times()
conc = gwt.output.concentration().get_data(totim=times[-1])
conc[conc < 1e30].min(), conc[conc < 1e30].max()

In [ ]:
c = gwt.output.concentration().get_data(totim=times[0])

### Plot settings

Shared figure size and contour styling used by the concentration figures below.

In [ ]:
six_panel_figsize = (17.15 / 2.541, 1.4 * 0.8333 * 17.15 / 2.541)
mosaic = [[0, 1, 2], [3, 4, "."]]

In [ ]:
contour_color = "white"
contour_style = "--"

sv_gwt_contour_dict = {
    "linewidths": 0.75,
    "colors": contour_color,
    "linestyles": contour_style,
}

### Concentration maps through time

Simulated concentration in each layer at the selected time, on a log color scale.

In [ ]:
times = gwt.output.concentration().get_times()

extent = gwt.modelgrid.extent
xc = lake_center_x  # cross-section through the center of the lake (A-A')

# reuse the six-panel layout but turn the empty slot into a colorbar axes
conc_mosaic = [[0, 1, 2], [3, 4, "cbar"]]


def plot_concentration(time_index):
    totim = times[time_index]
    conc = gwt.output.concentration().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            conc_mosaic,
            figsize=six_panel_figsize,
            layout="constrained",
        )
        fig.suptitle(f"Time = {totim:g} days (time index {time_index})")

        norm = mpl.colors.LogNorm(vmin=0.01, vmax=100)
        for k in range(nlay):
            ax = axd[k]
            ax.set_xlim(extent[:2])
            ax.set_ylim(extent[2:])
            ax.set_title(f"Layer {k + 1}")
            mm = flopy.plot.PlotMapView(
                model=gwt,
                ax=ax,
                extent=gwt.modelgrid.extent,
                layer=k,
            )
            mm.plot_ibound(color_vpt="blue", alpha=0.25)
            pa = mm.plot_array(
                conc,
                edgecolor="black",
                lw=0.25,
                masked_values=[0],
                norm=norm,
            )

            mm.plot_bc(package=gwf.sfr, color="cyan")
            mm.plot_bc(package=gwf.wel, color="red")

            c = conc[k]
            cmax = c[c < 1e30].max()
            contour_array = cmax >= 0.1
            if contour_array:
                cs = mm.contour_array(
                    conc,
                    # masked_values=[0],
                    levels=[0.01, 1, 10, 100],
                    **sv_gwt_contour_dict,
                )

                ax.clabel(
                    cs,
                    inline=True,
                    fmt="%g",
                    fontsize=8,
                    inline_spacing=0.5,
                    colors="white",
                )

            # show the cross-section location on the Layer 1 panel
            if k == 0:
                ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
                ax.text(
                    xc, extent[3], "A", color="red", fontweight="bold",
                    ha="center", va="bottom",
                )
                ax.text(
                    xc, extent[2], "A'", color="red", fontweight="bold",
                    ha="center", va="top",
                )

        # slim vertical colorbar centered in the empty mosaic panel
        axd["cbar"].axis("off")
        cax = inset_axes(
            axd["cbar"],
            width="25%",
            height="85%",
            loc="center",
        )
        ticks = [0.01, 0.1, 1, 10, 100]
        cb = fig.colorbar(pa, cax=cax, ticks=ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.minorticks_off()
        cb.set_label("Concentration")

    return fig


time_slider_view(plot_concentration, len(times))

### Concentration along section A–A' through time

Simulated concentration in cross section along A–A', shown only in the saturated part of each layer.

In [ ]:
# simulated concentration along section A-A' (center of the lake) through time
# (same settings as the concentration map slider figure above)
extent = gwt.modelgrid.extent
xc = lake_center_x  # cross-section through the center of the lake (A-A')
line = {"line": [(xc, extent[3]), (xc, extent[2])]}  # A (top) -> A' (bottom)

times = gwt.output.concentration().get_times()
conc_levels = [0.01, 1, 10, 100]
conc_ticks = [0.01, 0.1, 1, 10, 100]
xs_w = 17.15 / 2.541


def plot_concentration_xs(time_index):
    totim = times[time_index]
    conc = gwt.output.concentration().get_data(totim=totim)
    # head at the same time so concentrations are only shown in the
    # saturated portion of each layer
    head = gwf.output.head().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(xs_w, 0.5 * xs_w), layout="constrained")
        ax.set_title(
            f"Concentration along A-A', time = {totim:g} days (index {time_index})"
        )
        xs = flopy.plot.PlotCrossSection(model=gwt, ax=ax, line=line)

        norm = mpl.colors.LogNorm(vmin=0.01, vmax=100)
        pa = xs.plot_array(conc, head=head, masked_values=[0], norm=norm)
        xs.plot_grid(lw=0.3, color="0.5")
        xs.plot_ibound()

        cmax = conc[conc < 1e30].max()
        if cmax >= conc_levels[0]:
            cs = xs.contour_array(
                conc,
                head=head,
                levels=conc_levels,
                **sv_gwt_contour_dict,
            )
            ax.clabel(
                cs,
                inline=True,
                fmt="%g",
                fontsize=8,
                inline_spacing=0.5,
                colors=contour_color,
            )

        ax.set_xlabel("Distance along section (m)")
        ax.set_ylabel("Elevation (m)")

        # label the section endpoints to match the map figure
        ax.text(
            0.0, 1.01, "A", color="red", fontweight="bold",
            ha="left", va="bottom", transform=ax.transAxes,
        )
        ax.text(
            1.0, 1.01, "A'", color="red", fontweight="bold",
            ha="right", va="bottom", transform=ax.transAxes,
        )

        cb = fig.colorbar(pa, ax=ax, shrink=0.8, ticks=conc_ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.minorticks_off()
        cb.set_label("Concentration")

    return fig


time_slider_view(plot_concentration_xs, len(times))